# День 4 — Baseline без обучения трансформера

Цель: использовать замороженный DistilBERT для получения
CLS-эмбеддингов и обучить на них Logistic Regression.

In [1]:
from pathlib import Path

import pandas as pd
from datasets import load_dataset

C:\ProgramData\anaconda3\envs\transformers_overall\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data_dir = Path("../data")
baseline_output_dir = Path("../outputs/baseline")

data_dir.mkdir(parents=True, exist_ok=True)
baseline_output_dir.mkdir(parents=True, exist_ok=True)

print("Data directory:", data_dir.resolve())
print("Output directory:", baseline_output_dir.resolve())

Data directory: C:\Users\User\Projects\transformers_overall\data
Output directory: C:\Users\User\Projects\transformers_overall\outputs\baseline


In [3]:
sst2_train = load_dataset(
    "stanfordnlp/sst2",
    split="train",
)

print(sst2_train)
print(sst2_train.features)
print(sst2_train[0])

Dataset({
    features: ['idx', 'sentence', 'label'],
    num_rows: 67349
})
{'idx': Value('int32'), 'sentence': Value('string'), 'label': ClassLabel(names=['negative', 'positive'])}
{'idx': 0, 'sentence': 'hide new secretions from the parental units ', 'label': 0}


In [4]:
df = (
    sst2_train
    .to_pandas()[["sentence", "label"]]
    .rename(columns={"sentence": "text"})
)

df = df.dropna(subset=["text", "label"]).copy()

df["text"] = df["text"].astype(str).str.strip()

df = df[df["text"] != ""]

df = (
    df
    .drop_duplicates(subset=["text"])
    .reset_index(drop=True)
)

print("Размер после очистки:", df.shape)
display(df.head())

print("Пропуски:")
print(df.isna().sum())

print("\nРаспределение классов:")
print(df["label"].value_counts().sort_index())

print("\nНазвания колонок:")
print(df.columns.tolist())

Размер после очистки: (66978, 2)


,text,label
0,hide new secretions from the parental units,0
1,"contains no wit , only labored gags",0
2,that loves its characters and communicates som...,1
3,remains utterly satisfied to remain the same t...,0
4,on the worst revenge-of-the-nerds clichés the ...,0


Пропуски:
text     0
label    0
dtype: int64

Распределение классов:
label
0    29647
1    37331
Name: count, dtype: int64

Названия колонок:
['text', 'label']


In [5]:
samples_per_class = 1000
random_state = 42

negative_sample = df[df["label"] == 0].sample(
    n=samples_per_class,
    random_state=random_state,
)

positive_sample = df[df["label"] == 1].sample(
    n=samples_per_class,
    random_state=random_state,
)

sample_df = (
    pd.concat(
        [negative_sample, positive_sample],
        ignore_index=True,
    )
    .sample(frac=1, random_state=random_state)
    .reset_index(drop=True)
)

print("Размер sample:", sample_df.shape)
print(sample_df["label"].value_counts().sort_index())

display(sample_df.head())

Размер sample: (2000, 2)
label
0    1000
1    1000
Name: count, dtype: int64


,text,label
0,with vibrance and warmth,1
1,"stale ,",0
2,glides gracefully from male persona to female ...,1
3,", all-over-the-map movie would be a lot better...",0
4,the case for a strong education and good teach...,1


In [6]:
dataset_path = data_dir / "sst2_sample.csv"

sample_df.to_csv(
    dataset_path,
    index=False,
    encoding="utf-8",
)

print("Датасет сохранён:", dataset_path.resolve())
print("Файл существует:", dataset_path.exists())

Датасет сохранён: C:\Users\User\Projects\transformers_overall\data\sst2_sample.csv
Файл существует: True


In [7]:
loaded_df = pd.read_csv(dataset_path)

print("Загруженный размер:", loaded_df.shape)
display(loaded_df.head())

Загруженный размер: (2000, 2)


,text,label
0,with vibrance and warmth,1
1,"stale ,",0
2,glides gracefully from male persona to female ...,1
3,", all-over-the-map movie would be a lot better...",0
4,the case for a strong education and good teach...,1


In [8]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, y_train, y_test = train_test_split(
    sample_df["text"].tolist(),
    sample_df["label"].to_numpy(),
    test_size=0.2,
    stratify=sample_df["label"],
    random_state=42,
)

In [9]:
print("Train:", len(train_texts), y_train.shape)
print("Test:", len(test_texts), y_test.shape)

print("\nКлассы в train:")
print(pd.Series(y_train).value_counts().sort_index())

print("\nКлассы в test:")
print(pd.Series(y_test).value_counts().sort_index())

Train: 1600 (1600,)
Test: 400 (400,)

Классы в train:
0    800
1    800
Name: count, dtype: int64

Классы в test:
0    200
1    200
Name: count, dtype: int64


In [10]:
assert len(train_texts) == 1600
assert len(test_texts) == 400
assert set(y_train) == {0, 1}
assert set(y_test) == {0, 1}

In [11]:
import torch

from transformers import AutoModel, AutoTokenizer

model_name = "distilbert-base-uncased"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

encoder_model = AutoModel.from_pretrained(model_name)
encoder_model.to(device)
encoder_model.eval()

print("Модель:", model_name)
print("Устройство:", device)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3011.81it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель: distilbert-base-uncased
Устройство: cpu


In [12]:
def tokenize_texts(texts, max_length=128):
    tokens = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

    return {
        name: tensor.to(device)
        for name, tensor in tokens.items()
    }

In [13]:
test_batch = tokenize_texts([
    "This movie was excellent.",
    "A terrible and boring film.",
])

print("Input IDs shape:", test_batch["input_ids"].shape)
print("Attention mask shape:", test_batch["attention_mask"].shape)
print("Устройство тензоров:", test_batch["input_ids"].device)

Input IDs shape: torch.Size([2, 8])
Attention mask shape: torch.Size([2, 8])
Устройство тензоров: cpu


In [14]:
import os
import torch

print("Логических процессоров:", os.cpu_count())
print("PyTorch intra-op threads:", torch.get_num_threads())
print("PyTorch inter-op threads:", torch.get_num_interop_threads())

Логических процессоров: 16
PyTorch intra-op threads: 8
PyTorch inter-op threads: 8


In [15]:
import numpy as np


def get_cls_embeddings(texts, batch_size=32, max_length=128):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        tokens = tokenize_texts(
            batch_texts,
            max_length=max_length,
        )

        with torch.no_grad():
            outputs = encoder_model(**tokens)

        cls_embeddings = outputs.last_hidden_state[:, 0, :]

        all_embeddings.append(
            cls_embeddings.cpu().numpy()
        )

    return np.vstack(all_embeddings)

In [16]:
example_texts = [
    "This movie was absolutely amazing!",
    "Terrible movie, waste of time.",
    "Pretty good, I liked it.",
    "Boring and too long.",
]

example_embeddings = get_cls_embeddings(
    example_texts,
    batch_size=2,
)

print("Embeddings shape:", example_embeddings.shape)
print("Тип данных:", example_embeddings.dtype)

Embeddings shape: (4, 768)
Тип данных: float32


In [17]:
import time

start_time = time.time()

X_train = get_cls_embeddings(
    train_texts,
    batch_size=32,
    max_length=128,
)

X_test = get_cls_embeddings(
    test_texts,
    batch_size=32,
    max_length=128,
)

elapsed_time = time.time() - start_time

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print(f"Время получения эмбеддингов: {elapsed_time:.1f} секунд")

X_train shape: (1600, 768)
X_test shape: (400, 768)
Время получения эмбеддингов: 35.2 секунд


In [18]:
assert X_train.shape == (1600, 768)
assert X_test.shape == (400, 768)

assert X_train.dtype == np.float32
assert X_test.dtype == np.float32

assert np.isfinite(X_train).all()
assert np.isfinite(X_test).all()

In [19]:
import time

from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(
    max_iter=1000,
    n_jobs=-1,
    random_state=42,
)

start_time = time.time()

classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

training_time = time.time() - start_time

print(f"Время обучения классификатора: {training_time:.2f} секунд")
print("Количество предсказаний:", len(y_pred))
print("Первые 10 предсказаний:", y_pred[:10])
print("Первые 10 правильных меток:", y_test[:10])

Время обучения классификатора: 7.13 секунд
Количество предсказаний: 400
Первые 10 предсказаний: [0 0 1 0 0 1 1 0 0 1]
Первые 10 правильных меток: [0 0 1 0 0 1 1 0 0 1]


In [20]:
assert len(y_pred) == len(y_test)
assert set(y_pred).issubset({0, 1})

In [21]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

class_names = ["negative", "positive"]

report = classification_report(
    y_test,
    y_pred,
    target_names=class_names,
    digits=4,
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
)

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print("Classification report:")
print(report)

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

print("\nConfusion matrix:")
print(conf_matrix)

Classification report:
              precision    recall  f1-score   support

    negative     0.8579    0.8450    0.8514       200
    positive     0.8473    0.8600    0.8536       200

    accuracy                         0.8525       400
   macro avg     0.8526    0.8525    0.8525       400
weighted avg     0.8526    0.8525    0.8525       400

Accuracy: 0.8525
Macro F1: 0.8525

Confusion matrix:
[[169  31]
 [ 28 172]]


In [22]:
assert 0.0 <= accuracy <= 1.0
assert 0.0 <= macro_f1 <= 1.0
assert conf_matrix.shape == (2, 2)
assert conf_matrix.sum() == len(y_test)

print("Все проверки метрик пройдены.")

Все проверки метрик пройдены.


In [23]:
results_path = baseline_output_dir / "baseline_results.txt"

results_text = (
    "Transformers Day 4 — Baseline Results\n"
    "=====================================\n"
    f"Encoder: {model_name}\n"
    f"Train samples: {len(y_train)}\n"
    f"Test samples: {len(y_test)}\n"
    f"Embedding size: {X_train.shape[1]}\n"
    f"Accuracy: {accuracy:.4f}\n"
    f"Macro F1: {macro_f1:.4f}\n\n"
    "Classification report:\n"
    f"{report}\n"
    "Confusion matrix:\n"
    f"{conf_matrix}\n"
)

results_path.write_text(
    results_text,
    encoding="utf-8",
)

print("Результаты сохранены:", results_path.resolve())
print("Файл существует:", results_path.exists())

Результаты сохранены: C:\Users\User\Projects\transformers_overall\outputs\baseline\baseline_results.txt
Файл существует: True


In [24]:
print(results_path.read_text(encoding="utf-8"))

Transformers Day 4 — Baseline Results
Encoder: distilbert-base-uncased
Train samples: 1600
Test samples: 400
Embedding size: 768
Accuracy: 0.8525
Macro F1: 0.8525

Classification report:
              precision    recall  f1-score   support

    negative     0.8579    0.8450    0.8514       200
    positive     0.8473    0.8600    0.8536       200

    accuracy                         0.8525       400
   macro avg     0.8526    0.8525    0.8525       400
weighted avg     0.8526    0.8525    0.8525       400

Confusion matrix:
[[169  31]
 [ 28 172]]

